In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing Libs

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Utilizing GPU

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## Dataset Class

In [4]:
class AnimalDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

        # get unique labels and create mapping
        self.labels = sorted(self.data['label'].unique())
        self.label_to_idx = {label: idx for idx, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label']

        # load image
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label_idx = self.label_to_idx[label]
        return image, label_idx

## Transforms

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Evalutaion Function

In [6]:
def evaluate_model(model, data_loader, return_predictions=True):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    if return_predictions:
        return all_preds, all_labels
    else:
        acc = accuracy_score(all_labels, all_preds)
        return acc * 100

## Training Function

In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({'loss': running_loss/len(train_loader), 'acc': 100*correct/total})

        train_acc = 100 * correct / total

        # validation
        val_acc = evaluate_model(model, val_loader, return_predictions=False)
        print(f'Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    return model

## Calculate Metrics

In [8]:
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

## Training

In [9]:
base_path = '/content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv'
num_folds = 5

results = {
    'GoogLeNet': [],

}

for fold in range(1, num_folds + 1):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}/{num_folds}')
    print(f'{"="*60}')

    fold_path = os.path.join(base_path, f'fold_{fold}')

    train_csv = os.path.join(fold_path, 'train.csv')
    val_csv = os.path.join(fold_path, 'val.csv')
    test_csv = os.path.join(fold_path, 'test.csv')

    # create datasets
    train_dataset = AnimalDataset(train_csv, transform=train_transform)
    val_dataset = AnimalDataset(val_csv, transform=test_transform)
    test_dataset = AnimalDataset(test_csv, transform=test_transform)

    num_classes = len(train_dataset.labels)
    print(f'Number of classes: {num_classes}')
    print(f'Classes: {train_dataset.labels}')

    # create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

    # train googlenet
    print(f'\nTraining googlenet on Fold {fold+1}...')
    googlenet = models.googlenet(pretrained=True)

    # Freeze all parameters in the network
    for param in googlenet.parameters():
        param.requires_grad = False

    # Get the number of input features for the final fully connected layer
    num_ftrs = googlenet.fc.in_features # Corrected from _fc to fc
    # Replace the final fully connected layer
    googlenet.fc = nn.Linear(num_ftrs, num_classes) # Corrected from _fc to fc

    googlenet = googlenet.to(device)

    # Only parameters that have requires_grad=True (the new fc layer) will be optimized
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(googlenet.parameters(), lr=0.0001)

    googlenet = train_model(googlenet, train_loader, val_loader, criterion, optimizer, epochs=5)

    preds, labels = evaluate_model(googlenet, test_loader)
    googlenet_metrics = calculate_metrics(labels, preds)
    results['GoogLeNet'].append(googlenet_metrics)

    print(f'\ngooglenet Results (Fold {fold+1}):')
    for metric, value in googlenet_metrics.items():
        print(f'{metric}: {value:.4f}')



FOLD 1/5
Number of classes: 64
Classes: ['antelope', 'bear', 'beaver', 'bee', 'bison', 'blackbird', 'buffalo', 'butterfly', 'camel', 'cat', 'cheetah', 'chimpanzee', 'chinchilla', 'cow', 'crab', 'crocodile', 'deer', 'dog', 'dolphin', 'donkey', 'duck', 'eagle', 'elephant', 'falcon', 'ferret', 'flamingo', 'fox', 'frog', 'giraffe', 'goat', 'goose', 'gorilla', 'grasshopper', 'hawk', 'hedgehog', 'hippopotamus', 'hyena', 'iguana', 'jaguar', 'kangaroo', 'koala', 'lemur', 'leopard', 'lizard', 'lynx', 'mole', 'mongoose', 'ostrich', 'otter', 'owl', 'panda', 'peacock', 'penguin', 'porcupine', 'raccoon', 'seal', 'sheep', 'snail', 'snake', 'spider', 'squid', 'walrus', 'whale', 'wolf']

Training googlenet on Fold 2...
Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


100%|██████████| 49.7M/49.7M [00:00<00:00, 121MB/s]
Epoch 1/5: 100%|██████████| 288/288 [36:37<00:00,  7.63s/it, loss=3.19, acc=56.5]


Epoch 1: Train Acc = 56.51%, Val Acc = 91.13%


Epoch 2/5: 100%|██████████| 288/288 [02:35<00:00,  1.86it/s, loss=1.66, acc=93.2]


Epoch 2: Train Acc = 93.15%, Val Acc = 96.13%


...

Epoch 3: Train Acc = 96.40%, Val Acc = 97.52%


Epoch 4/5: 100%|██████████| 288/288 [02:30<00:00,  1.92it/s, loss=0.568, acc=97.1]


Epoch 4: Train Acc = 97.13%, Val Acc = 97.96%


Epoch 5/5: 100%|██████████| 288/288 [02:28<00:00,  1.94it/s, loss=0.404, acc=97.6]


Epoch 5: Train Acc = 97.60%, Val Acc = 98.04%

googlenet Results (Fold 6):
Accuracy: 0.9850
Precision: 0.9857
Recall: 0.9850
F1-Score: 0.9850


## Results

In [12]:
print('\n' + '='*70)
print('FINAL RESULTS ACROSS ALL FOLDS')
print('='*70)

model_name = 'GoogLeNet' # Corrected: use the string key 'GoogLeNet'
print(f'\n{model_name}:')
print('-'*70)

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for metric in metrics_names:
      values = [fold[metric] for fold in results[model_name]]
      mean_val = np.mean(values)
      std_val = np.std(values)
      print(f'{metric:12s}: {mean_val:.4f} \u00b1 {std_val:.4f}')


FINAL RESULTS ACROSS ALL FOLDS

GoogLeNet:
----------------------------------------------------------------------
Accuracy    : 0.9831 ± 0.0022
Precision   : 0.9838 ± 0.0020
Recall      : 0.9831 ± 0.0022
F1-Score    : 0.9831 ± 0.0022
